# 📊 Embryoscope & Clinisys Combined Metrics: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database and **AWS Athena (Production)** to execute data quality and validation queries on the `embryoscope_clinisys_combined` table.

It reconciles:
1. **Total Row Counts** (Local vs Cloud)
2. **Matched Records Counts** (Local vs Cloud)
3. **Clinisys Only / Unmatched Records**
4. **Match Rates** (vs Clinisys and vs Embryoscope)
5. **Yearly Match Rates Breakdown** (based on Embryoscope fertilization year)
6. **Location Breakdown** (based on clinic locations)
7. **Discrepancy Root Cause Analysis** (investigating `prontuario` filling rate in base tables)

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Connections setup
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print("Connecting to AWS Athena...")
try:
    ath_con = connect(
        region_name="sa-east-1",
        work_group="datalake-admins"
    )
    ath_cur = ath_con.cursor()
    print("Successfully connected to AWS Athena.")
except Exception as e:
    print(f"Warning: Could not connect to Athena: {e}")
    ath_cur = None

## 🔍 Part 1: DuckDB Queries (Local Combined Table)

Running local metrics on `gold.embryoscope_clinisys_combined` in DuckDB.

In [ ]:
# Get base table counts dynamically
duck_emb_count = duck_con.execute("SELECT COUNT(*) FROM gold.embryoscope_embrioes").fetchone()[0]
duck_clin_count = duck_con.execute("SELECT COUNT(*) FROM gold.clinisys_embrioes").fetchone()[0]
duck_emb_count_2023 = duck_con.execute("SELECT COUNT(*) FROM gold.embryoscope_embrioes WHERE embryo_FertilizationTime >= '2023-01-01'").fetchone()[0]
duck_clin_count_2023 = duck_con.execute("SELECT COUNT(*) FROM gold.clinisys_embrioes WHERE micro_Data_DL >= '2023-01-01'").fetchone()[0]

# 1. DuckDB Overall
duck_overall_q = f"""
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(CASE WHEN embryo_EmbryoID IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) as matched_records,
    COUNT(CASE WHEN embryo_EmbryoID IS NULL AND oocito_id IS NOT NULL THEN 1 END) as clinisys_only_records,
    ROUND(COUNT(CASE WHEN embryo_EmbryoID IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) * 100.0 / {duck_clin_count}, 2) as match_rate_vs_clinisys,
    ROUND(COUNT(CASE WHEN embryo_EmbryoID IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) * 100.0 / {duck_emb_count}, 2) as match_rate_vs_embryoscope
FROM gold.embryoscope_clinisys_combined;
"""
df_duck_overall = duck_con.execute(duck_overall_q).df()
print("--- DuckDB Overall Totals ---")
display(df_duck_overall)

# 2. DuckDB 2023+ Overall
duck_overall_2023_q = f"""
SELECT 
    'TOTAL 2023+' as grouping_type,
    '2023_ONWARDS' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(CASE WHEN embryo_EmbryoID IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) as matched_records,
    COUNT(CASE WHEN embryo_EmbryoID IS NULL AND oocito_id IS NOT NULL THEN 1 END) as clinisys_only_records,
    ROUND(COUNT(CASE WHEN embryo_EmbryoID IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) * 100.0 / {duck_clin_count_2023}, 2) as match_rate_vs_clinisys,
    ROUND(COUNT(CASE WHEN embryo_EmbryoID IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) * 100.0 / {duck_emb_count_2023}, 2) as match_rate_vs_embryoscope
FROM gold.embryoscope_clinisys_combined
WHERE micro_Data_DL >= '2023-01-01';
"""
df_duck_overall_2023 = duck_con.execute(duck_overall_2023_q).df()
print("\n--- DuckDB 2023+ Totals ---")
display(df_duck_overall_2023)

# 3. DuckDB Yearly Breakdown
duck_yearly_q = """
WITH emb_yearly AS (
    SELECT 
        EXTRACT(YEAR FROM CAST(embryo_FertilizationTime AS DATE)) as year_val,
        COUNT(*) as total_emb
    FROM gold.embryoscope_embrioes
    GROUP BY 1
),
matched_yearly AS (
    SELECT 
        EXTRACT(YEAR FROM CAST(embryo_FertilizationTime AS DATE)) as year_val,
        COUNT(DISTINCT embryo_EmbryoID) as matched_emb
    FROM gold.embryoscope_clinisys_combined
    WHERE embryo_EmbryoID IS NOT NULL
    GROUP BY 1
)
SELECT 
    e.year_val as year,
    e.total_emb as total_embryoscope,
    COALESCE(m.matched_emb, 0) as matched,
    ROUND((COALESCE(m.matched_emb, 0) * 100.0) / e.total_emb, 2) as match_rate
FROM emb_yearly e
LEFT JOIN matched_yearly m ON e.year_val = m.year_val
ORDER BY e.year_val;
"""
df_duck_yearly = duck_con.execute(duck_yearly_q).df()
print("\n--- DuckDB Yearly Breakdown ---")
display(df_duck_yearly)

# 4. DuckDB Location Breakdown
duck_location_q = """
SELECT 
    COALESCE(patient_unit_huntington, 'Clinisys Only / Unmatched') as location,
    COUNT(*) as total_rows,
    COUNT(CASE WHEN embryo_EmbryoID IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) as matched_records,
    COUNT(CASE WHEN embryo_EmbryoID IS NULL AND oocito_id IS NOT NULL THEN 1 END) as clinisys_only_records
FROM gold.embryoscope_clinisys_combined
GROUP BY 1
ORDER BY 2 DESC;
"""
df_duck_location = duck_con.execute(duck_location_q).df()
print("\n--- DuckDB Location Breakdown ---")
display(df_duck_location)

## ☁️ Part 2: AWS Athena Queries (Production Combined Table)

Running metrics on `gold_huntington_prod.embryoscope_clinisys_combined` in AWS Athena.

In [ ]:
if ath_cur is not None:
    # Get base table counts dynamically
    ath_cur.execute("SELECT COUNT(*) FROM gold_huntington_prod.embryoscope_embrioes")
    ath_emb_count = ath_cur.fetchone()[0]
    
    ath_cur.execute("SELECT COUNT(*) FROM gold_huntington_prod.clinisys_embrioes")
    ath_clin_count = ath_cur.fetchone()[0]
    
    ath_cur.execute("SELECT COUNT(*) FROM gold_huntington_prod.embryoscope_embrioes WHERE embryo_fertilization_time >= TIMESTAMP '2023-01-01 00:00:00'")
    ath_emb_count_2023 = ath_cur.fetchone()[0]
    
    ath_cur.execute("SELECT COUNT(*) FROM gold_huntington_prod.clinisys_embrioes WHERE micro_data_dl >= DATE '2023-01-01'")
    ath_clin_count_2023 = ath_cur.fetchone()[0]
    
    # 1. Athena Overall
    ath_overall_q = f"""
    SELECT 
        'TOTAL' as grouping_type,
        'ALL' as grouping_value,
        COUNT(*) as total_rows,
        COUNT(CASE WHEN embryo_embryo_id IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) as matched_records,
        COUNT(CASE WHEN embryo_embryo_id IS NULL AND oocito_id IS NOT NULL THEN 1 END) as clinisys_only_records,
        ROUND(COUNT(CASE WHEN embryo_embryo_id IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) * 100.0 / {ath_clin_count}, 2) as match_rate_vs_clinisys,
        ROUND(COUNT(CASE WHEN embryo_embryo_id IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) * 100.0 / {ath_emb_count}, 2) as match_rate_vs_embryoscope
    FROM gold_huntington_prod.embryoscope_clinisys_combined
    """
    ath_cur.execute(ath_overall_q)
    df_ath_overall = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    print("--- Athena Overall Totals ---")
    display(df_ath_overall)

    # 2. Athena 2023+ Overall
    ath_overall_2023_q = f"""
    SELECT 
        'TOTAL 2023+' as grouping_type,
        '2023_ONWARDS' as grouping_value,
        COUNT(*) as total_rows,
        COUNT(CASE WHEN embryo_embryo_id IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) as matched_records,
        COUNT(CASE WHEN embryo_embryo_id IS NULL AND oocito_id IS NOT NULL THEN 1 END) as clinisys_only_records,
        ROUND(COUNT(CASE WHEN embryo_embryo_id IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) * 100.0 / {ath_clin_count_2023}, 2) as match_rate_vs_clinisys,
        ROUND(COUNT(CASE WHEN embryo_embryo_id IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) * 100.0 / {ath_emb_count_2023}, 2) as match_rate_vs_embryoscope
    FROM gold_huntington_prod.embryoscope_clinisys_combined
    WHERE micro_data_dl >= DATE '2023-01-01'
    """
    ath_cur.execute(ath_overall_2023_q)
    df_ath_overall_2023 = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    print("\n--- Athena 2023+ Totals ---")
    display(df_ath_overall_2023)

    # 3. Athena Yearly Breakdown
    ath_yearly_q = """
    WITH emb_yearly AS (
        SELECT 
            EXTRACT(YEAR FROM embryo_fertilization_time) as year_val,
            COUNT(*) as total_emb
        FROM gold_huntington_prod.embryoscope_embrioes
        GROUP BY 1
    ),
    matched_yearly AS (
        SELECT 
            EXTRACT(YEAR FROM embryo_fertilization_time) as year_val,
            COUNT(DISTINCT embryo_embryo_id) as matched_emb
        FROM gold_huntington_prod.embryoscope_clinisys_combined
        WHERE embryo_embryo_id IS NOT NULL
        GROUP BY 1
    )
    SELECT 
        e.year_val as year,
        e.total_emb as total_embryoscope,
        COALESCE(m.matched_emb, 0) as matched,
        ROUND((COALESCE(m.matched_emb, 0) * 100.0) / e.total_emb, 2) as match_rate
    FROM emb_yearly e
    LEFT JOIN matched_yearly m ON e.year_val = m.year_val
    ORDER BY e.year_val
    """
    ath_cur.execute(ath_yearly_q)
    df_ath_yearly = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    df_ath_yearly['year'] = df_ath_yearly['year'].astype(float)
    print("\n--- Athena Yearly Breakdown ---")
    display(df_ath_yearly)

    # 4. Athena Location Breakdown
    ath_location_q = """
    SELECT 
        COALESCE(embryo_unit_huntington, 'Clinisys Only / Unmatched') as location,
        COUNT(*) as total_rows,
        COUNT(CASE WHEN embryo_embryo_id IS NOT NULL AND oocito_id IS NOT NULL THEN 1 END) as matched_records,
        COUNT(CASE WHEN embryo_embryo_id IS NULL AND oocito_id IS NOT NULL THEN 1 END) as clinisys_only_records
    FROM gold_huntington_prod.embryoscope_clinisys_combined
    GROUP BY 1
    ORDER BY 2 DESC
    """
    ath_cur.execute(ath_location_q)
    df_ath_location = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    print("\n--- Athena Location Breakdown ---")
    display(df_ath_location)
else:
    print("Athena cursor is None. Skipping queries.")

## ⚖️ Part 3: Side-by-Side Reconciliation Tables

Merging and displaying the local (DuckDB) and cloud (Athena) metrics side-by-side to highlight differences.

In [ ]:
if ath_cur is not None:
    # 1. Overall comparison
    df_duck_all = pd.concat([df_duck_overall, df_duck_overall_2023], ignore_index=True)
    df_ath_all = pd.concat([df_ath_overall, df_ath_overall_2023], ignore_index=True)
    
    df_comp_overall = pd.merge(
        df_duck_all.rename(columns={
            'total_rows': 'duck_total_rows',
            'matched_records': 'duck_matched',
            'clinisys_only_records': 'duck_clinisys_only',
            'match_rate_vs_clinisys': 'duck_match_rate_vs_clin',
            'match_rate_vs_embryoscope': 'duck_match_rate_vs_emb'
        })[['grouping_type', 'duck_total_rows', 'duck_matched', 'duck_clinisys_only', 'duck_match_rate_vs_clin', 'duck_match_rate_vs_emb']],
        df_ath_all.rename(columns={
            'total_rows': 'ath_total_rows',
            'matched_records': 'ath_matched',
            'clinisys_only_records': 'ath_clinisys_only',
            'match_rate_vs_clinisys': 'ath_match_rate_vs_clin',
            'match_rate_vs_embryoscope': 'ath_match_rate_vs_emb'
        })[['grouping_type', 'ath_total_rows', 'ath_matched', 'ath_clinisys_only', 'ath_match_rate_vs_clin', 'ath_match_rate_vs_emb']],
        on='grouping_type'
    )
    print("--- Side-by-side Overall Comparison ---")
    display(df_comp_overall)

    # 2. Yearly breakdown comparison
    df_duck_yearly['year'] = df_duck_yearly['year'].astype(float)
    df_ath_yearly['year'] = df_ath_yearly['year'].astype(float)
    
    df_comp_yearly = pd.merge(
        df_duck_yearly.rename(columns={
            'total_embryoscope': 'duck_total_embryo',
            'matched': 'duck_matched',
            'match_rate': 'duck_match_rate'
        }),
        df_ath_yearly.rename(columns={
            'total_embryoscope': 'ath_total_embryo',
            'matched': 'ath_matched',
            'match_rate': 'ath_match_rate'
        }),
        on='year',
        how='outer'
    ).sort_values('year')
    print("\n--- Side-by-side Yearly Match Rate Comparison ---")
    display(df_comp_yearly)

    # 3. Location breakdown comparison
    df_duck_location['location'] = df_duck_location['location'].str.strip()
    df_ath_location['location'] = df_ath_location['location'].str.strip()
    
    df_comp_location = pd.merge(
        df_duck_location.rename(columns={
            'total_rows': 'duck_rows',
            'matched_records': 'duck_matched',
            'clinisys_only_records': 'duck_clinisys_only'
        }),
        df_ath_location.rename(columns={
            'total_rows': 'ath_rows',
            'matched_records': 'ath_matched',
            'clinisys_only_records': 'ath_clinisys_only'
        }),
        on='location',
        how='outer'
    )
    print("\n--- Side-by-side Location Breakdown Comparison ---")
    display(df_comp_location)

## 🔍 Part 4: Root Cause Discrepancy Analysis

Investigating why local DuckDB matches 108,906 records while AWS Athena only matches 40,203 records.
We evaluate the `prontuario` (medical record number) filling quality in the `embryoscope_embrioes` base table in both environments.

In [ ]:
# 1. Check prontuario valid rates in base tables
print("=== BASE TABLE PRONTUARIO FILLING QUALITY ===")

duck_pront_q = """
SELECT 
    COUNT(*) as total_rows,
    COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) as valid_prontuarios,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as valid_pront_pct
FROM gold.embryoscope_embrioes
"""
df_duck_pront = duck_con.execute(duck_pront_q).df()
print("Local DuckDB base embryoscope table prontuario quality:")
display(df_duck_pront)

if ath_cur is not None:
    ath_pront_q = """
    SELECT 
        COUNT(*) as total_rows,
        COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) as valid_prontuarios,
        ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as valid_pront_pct
    FROM gold_huntington_prod.embryoscope_embrioes
    """
    ath_cur.execute(ath_pront_q)
    df_ath_pront = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    print("\nAthena Production base embryoscope table prontuario quality:")
    display(df_ath_pront)
    
    # 2. Find sample of records that matched locally in DuckDB but did NOT match in Athena
    print("\n=== SAMPLE DISCREPANT RECORDS ===")
    print("Finding 5 embryos that are matched locally (have embryo_EmbryoID) but are NOT matched in Athena (embryo_embryo_id IS NULL):")
    
    sample_discrepant_q = """
        SELECT DISTINCT embryo_EmbryoID
        FROM gold.embryoscope_clinisys_combined
        WHERE embryo_EmbryoID IS NOT NULL
        LIMIT 200
    """
    duck_matched_ids = set(duck_con.execute(sample_discrepant_q).df()['embryo_EmbryoID'].tolist())
    
    # Get the matched ones in Athena for the same subset
    placeholders = ", ".join([f"'{eid}'" for eid in duck_matched_ids])
    ath_check_q = f"""
        SELECT embryo_embryo_id 
        FROM gold_huntington_prod.embryoscope_clinisys_combined 
        WHERE embryo_embryo_id IN ({placeholders})
    """
    ath_cur.execute(ath_check_q)
    ath_matched_ids = set(r[0] for r in ath_cur.fetchall())
    
    discrepant_ids = sorted(list(duck_matched_ids - ath_matched_ids))[:5]
    print(f"Sample discrepant embryo IDs: {discrepant_ids}")
    
    if discrepant_ids:
        # Get details from DuckDB
        disc_placeholders = ", ".join([f"'{eid}'" for eid in discrepant_ids])
        duck_details = duck_con.execute(f"""
            SELECT 
                oocito_id, micro_prontuario, micro_Data_DL,
                embryo_EmbryoID, embryo_FertilizationTime, prontuario
            FROM gold.embryoscope_clinisys_combined
            WHERE embryo_EmbryoID IN ({disc_placeholders})
        """).df()
        print("\nDetails in local DuckDB:")
        display(duck_details)
        
        # Get details from Athena for the same oocito_ids
        oocito_ids_str = ", ".join([str(oid) for oid in duck_details['oocito_id'].tolist()])
        ath_details_q = f"""
            SELECT 
                oocito_id, micro_prontuario, micro_data_dl,
                embryo_embryo_id, embryo_fertilization_time, embryo_prontuario as prontuario
            FROM gold_huntington_prod.embryoscope_clinisys_combined
            WHERE oocito_id IN ({oocito_ids_str})
        """
        ath_cur.execute(ath_details_q)
        df_ath_details = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
        print("\nDetails in Athena for the same oocito_ids:")
        display(df_ath_details)
